# 04 - Statistical Analysis

This notebook performs statistical analysis on the Amsterdam Inside Airbnb dataset.

The goal is to test whether important market differences observed during EDA are statistically meaningful.

Main analysis areas:
- Price differences across room types
- Price differences across host portfolio segments
- Occupancy proxy differences across room types
- Review score differences across room types
- Correlation significance between price and key listing features

In [2]:
%pip install scipy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/37.3 MB ? eta -:--:--
    --------------------------------------- 0.5/37.3 MB 995.0 kB/s eta 0:00:37
   - -------------------------------------- 1.8/37.3 MB 2.7 MB/s eta 0:00:14
   -- ------------------------------------- 2.6/37.3 MB 3.1 MB/s eta 0:00:12
   ---- ----------------------------------- 3.9/37.3 MB 3.7 MB/s eta 0:00:09
   ----- ---------------------------------- 5.2/37.3 MB 4.2 MB/s eta 0:00:08
   ------- -------------------------------- 6.8/37.3 MB 4.7 MB/s eta 0:00:07
   --------- ------------------------------ 8.7/37.3 MB 5.2 MB/s eta 0:00:06
   ---------- ----------------------------- 10.2/37.3 MB 5.4 MB/s eta 0:00:05
   ------------ --------------------------- 11.8/37.3 MB 5.7 MB/s eta 0:00:05
   ------------

In [3]:
import scipy
print("SciPy version:", scipy.__version__)

SciPy version: 1.18.0


In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path("..")

WAREHOUSE_PATH = PROJECT_ROOT / "warehouse"
DUCKDB_DATABASE_PATH = WAREHOUSE_PATH / "airbnb_market.duckdb"

STAT_REPORTS_PATH = PROJECT_ROOT / "reports" / "statistical_analysis"
STAT_REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("DuckDB database exists:", DUCKDB_DATABASE_PATH.exists())
print("Statistical reports path:", STAT_REPORTS_PATH)

DuckDB database exists: True
Statistical reports path: ..\reports\statistical_analysis


In [4]:
conn = duckdb.connect(
    database=str(DUCKDB_DATABASE_PATH),
    read_only=True
)

print("Connected to DuckDB in read-only mode.")

Connected to DuckDB in read-only mode.


In [5]:
listing_master_df = conn.execute("""
SELECT *
FROM listing_master_final
""").fetchdf()

print("listing_master_df shape:", listing_master_df.shape)
listing_master_df.head()

listing_master_df shape: (10465, 101)


,listing_id,listing_name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,city,country,snapshot_date,host_is_superhost,host_response_time,host_response_rate,host_acceptance_rate,host_listings_count,host_total_listings_count,property_type,accommodates,bathrooms_text,bedrooms,beds,amenities,amenities_count,instant_bookable,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,host_since,host_identity_verified,bathrooms,availability_30,availability_60,availability_90,number_of_reviews_l30d,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,calendar_days,available_days,unavailable_days,weekend_available_days,weekend_total_days,weekday_available_days,weekday_total_days,avg_minimum_nights,median_minimum_nights,avg_maximum_nights,median_maximum_nights,availability_rate,occupancy_proxy,weekend_availability_rate,weekday_availability_rate,listing_price_available_flag,price_numeric,unavailable_days_numeric,estimated_revenue_proxy,host_portfolio_segment,availability_segment,detailed_review_count,unique_reviewer_count,first_detailed_review_date,latest_detailed_review_date,reviews_with_comments,avg_comment_length,median_comment_length,detailed_reviews_last_365d,comment_coverage_rate,review_history_days,review_history_years,avg_reviews_per_year,has_reviews,neighbourhood_listing_count,neighbourhood_unique_hosts,neighbourhood_avg_price,neighbourhood_median_price,neighbourhood_min_price,neighbourhood_max_price,neighbourhood_avg_availability_rate,neighbourhood_avg_occupancy_proxy,neighbourhood_avg_review_score,neighbourhood_total_reviews,neighbourhood_avg_reviews_per_listing,neighbourhood_total_revenue_proxy,neighbourhood_avg_revenue_proxy,neighbourhood_dominant_room_type
0,28871,Comfortable double room,124245,1462510208428038912,Edwin,<NA>,Centrum-West,52.367750,4.89092,Private room,94.0,1.0,799,2026-06-01,4.15,2.0,11,88,0363 607B EA74 0BD8 2F6F,Amsterdam,Netherlands,2026-06-15,True,<NA>,<NA>,<NA>,2.0,NaN,Private room in rental unit,2.0,1 shared bath,NaN,NaN,"[""Heating"", ""Fire extinguisher"", ""Hot water"", ...",18.0,<NA>,4.86,4.89,4.84,4.94,4.94,4.93,4.83,NaT,True,NaN,1.0,1.0,1.0,5.0,95.0,255.0,23970.0,365,11,354,5,104,6,261,1.994521,2.0,730.0,730.0,0.0301,0.9699,0.0481,0.0230,True,94.0,354,33276.0,Small portfolio host,Low availability,799,796,2010-08-22,2026-06-01,799,202.556946,163.0,89,1.0,5776.0,15.81,50.54,True,1217,923,398.36,290.0,39.0,11412.0,0.34,0.66,4.82,116784,95.96,65092219.0,74221.46,Entire home/apt
1,29051,Comfortable single / double room,124245,1462510208428038912,Edwin,<NA>,Centrum-Oost,52.365840,4.89111,Private room,NaN,1.0,906,2026-06-01,4.88,2.0,2,82,0363 607B EA74 0BD8 2F6F,Amsterdam,Netherlands,2026-06-15,True,<NA>,<NA>,<NA>,2.0,NaN,Private room in condo,2.0,1 shared bath,NaN,NaN,"[""Heating"", ""Fire extinguisher"", ""Hot water"", ...",18.0,<NA>,4.82,4.88,4.83,4.93,4.93,4.88,4.79,NaT,True,NaN,0.0,2.0,2.0,6.0,85.0,255.0,NaN,365,2,363,1,104,1,261,1.980822,2.0,730.0,730.0,0.0055,0.9945,0.0096,0.0038,False,NaN,363,NaN,Small portfolio host,Low availability,906,894,2011-03-16,2026-06-01,906,241.263797,183.5,82,1.0,5570.0,15.25,59.41,True,933,708,350.41,282.5,39.0,2477.0,0.36,0.64,4.81,85025,91.13,44237061.0,65246.40,Entire home/apt
2,44129,Luxury design with canal view,187728,1462512125293144576,Tanya,<NA>,Centrum-West,52.382110,4.88630,Entire home/apt,314.0,3.0,186,2026-04-30,0.96,4.0,3,5,03635399E87602900F47,Amsterdam,Netherlands,2026-06-15,True,<NA>,<NA>,<NA>,6.0,NaN,Entire rental unit,2.0,1.5 baths,1.0,NaN,"[""Smoking allowed"", ""Extra pillows and blanket...",32.0,<NA>,4.88,4.81,4.88,4.89,4.89,4.95,4.59,NaT,True,NaN,3.0,3.0,3.0,0.0,1.0,39.0,12233.0,365,3,362,1,104,2,261,4.983562,5.0,1125.0,1125.0,0.00

In [6]:
stats_price_df = listing_master_df[
    listing_master_df["price_numeric"].notna() &
    (listing_master_df["price_numeric"] > 0)
].copy()

print("Total listings:", len(listing_master_df))
print("Listings with valid price:", len(stats_price_df))
print("Missing/invalid price listings:", len(listing_master_df) - len(stats_price_df))

Total listings: 10465
Listings with valid price: 6471
Missing/invalid price listings: 3994


## Statistical Test Plan

The dataset contains skewed price distributions and visible outliers.  
Because of this, non-parametric tests are preferred for many comparisons.

Planned tests:
1. Kruskal-Wallis test for price differences across room types.
2. Kruskal-Wallis test for price differences across host portfolio segments.
3. Kruskal-Wallis test for occupancy proxy differences across room types.
4. Spearman correlation tests for price and selected numeric features.

Kruskal-Wallis is used instead of one-way ANOVA because price is not normally distributed and contains outliers.

## 1. Price Differences Across Room Types

This test checks whether listing prices are significantly different across room types.

A Kruskal-Wallis test is used because price is right-skewed and contains outliers.  
This non-parametric test compares whether price distributions differ across more than two independent groups.

In [7]:
room_type_price_summary = (
    stats_price_df
    .groupby("room_type")
    .agg(
        listing_count=("price_numeric", "count"),
        mean_price=("price_numeric", "mean"),
        median_price=("price_numeric", "median"),
        min_price=("price_numeric", "min"),
        max_price=("price_numeric", "max")
    )
    .reset_index()
)

room_type_price_summary["mean_price"] = room_type_price_summary["mean_price"].round(2)
room_type_price_summary["median_price"] = room_type_price_summary["median_price"].round(2)

room_type_price_summary

,room_type,listing_count,mean_price,median_price,min_price,max_price
0,Entire home/apt,4771,394.62,331.0,26.0,11412.0
1,Hotel room,26,249.85,227.5,69.0,700.0
2,Private room,1653,203.67,171.0,15.0,1712.0
3,Shared room,21,80.10,52.0,31.0,681.0


In [8]:
room_type_groups = []

for room_type in stats_price_df["room_type"].dropna().unique():
    group_prices = stats_price_df[
        stats_price_df["room_type"] == room_type
    ]["price_numeric"].dropna()
    
    room_type_groups.append(group_prices)

kruskal_statistic, kruskal_p_value = stats.kruskal(*room_type_groups)

room_type_price_test_result = pd.DataFrame({
    "test_name": ["Kruskal-Wallis"],
    "comparison": ["Price by room type"],
    "test_statistic": [kruskal_statistic],
    "p_value": [kruskal_p_value]
})

room_type_price_test_result

,test_name,comparison,test_statistic,p_value
0,Kruskal-Wallis,Price by room type,1839.304703,0.0


In [9]:
n = len(stats_price_df)
k = stats_price_df["room_type"].nunique()

epsilon_squared = (kruskal_statistic - k + 1) / (n - k)

room_type_price_test_result["epsilon_squared"] = round(epsilon_squared, 4)

room_type_price_test_result

,test_name,comparison,test_statistic,p_value,epsilon_squared
0,Kruskal-Wallis,Price by room type,1839.304703,0.0,0.284


In [10]:
room_type_price_summary_output_path = STAT_REPORTS_PATH / "room_type_price_summary.csv"
room_type_price_test_output_path = STAT_REPORTS_PATH / "room_type_price_kruskal_test.csv"

room_type_price_summary.to_csv(room_type_price_summary_output_path, index=False)
room_type_price_test_result.to_csv(room_type_price_test_output_path, index=False)

print(f"Room type price summary saved to: {room_type_price_summary_output_path}")
print(f"Room type price test result saved to: {room_type_price_test_output_path}")

Room type price summary saved to: ..\reports\statistical_analysis\room_type_price_summary.csv
Room type price test result saved to: ..\reports\statistical_analysis\room_type_price_kruskal_test.csv


### Interpretation

The Kruskal-Wallis test was used to evaluate whether listing prices differ across room types.

If the p-value is below 0.05, the result suggests that at least one room type has a significantly different price distribution compared with the others.

This test does not identify exactly which room types differ from each other.  
Pairwise post-hoc testing would be required for that, but the result provides statistical support for the EDA finding that room type is associated with pricing differences.

### Pairwise Room Type Price Comparisons

Because the Kruskal-Wallis test showed a statistically significant price difference across room types, pairwise Mann-Whitney U tests are used to identify which room type pairs differ.

Bonferroni correction is applied to control the risk of false positives from multiple comparisons.

In [11]:
from itertools import combinations

room_types = stats_price_df["room_type"].dropna().unique()

pairwise_results = []

for room_type_1, room_type_2 in combinations(room_types, 2):
    group_1 = stats_price_df[
        stats_price_df["room_type"] == room_type_1
    ]["price_numeric"].dropna()
    
    group_2 = stats_price_df[
        stats_price_df["room_type"] == room_type_2
    ]["price_numeric"].dropna()
    
    u_statistic, p_value = stats.mannwhitneyu(
        group_1,
        group_2,
        alternative="two-sided"
    )
    
    n1 = len(group_1)
    n2 = len(group_2)
    
    rank_biserial_correlation = (2 * u_statistic) / (n1 * n2) - 1
    
    pairwise_results.append({
        "group_1": room_type_1,
        "group_2": room_type_2,
        "group_1_count": n1,
        "group_2_count": n2,
        "group_1_median_price": group_1.median(),
        "group_2_median_price": group_2.median(),
        "median_difference_group_1_minus_group_2": group_1.median() - group_2.median(),
        "u_statistic": u_statistic,
        "p_value": p_value,
        "rank_biserial_correlation": rank_biserial_correlation
    })

pairwise_room_type_price_tests = pd.DataFrame(pairwise_results)

# Bonferroni correction
number_of_tests = len(pairwise_room_type_price_tests)

pairwise_room_type_price_tests["p_value_bonferroni"] = (
    pairwise_room_type_price_tests["p_value"] * number_of_tests
).clip(upper=1)

pairwise_room_type_price_tests["significant_after_bonferroni"] = (
    pairwise_room_type_price_tests["p_value_bonferroni"] < 0.05
)

# Round numeric columns for readability
round_columns = [
    "group_1_median_price",
    "group_2_median_price",
    "median_difference_group_1_minus_group_2",
    "u_statistic",
    "p_value",
    "p_value_bonferroni",
    "rank_biserial_correlation"
]

for column in round_columns:
    pairwise_room_type_price_tests[column] = pairwise_room_type_price_tests[column].round(6)

pairwise_room_type_price_tests

,group_1,group_2,group_1_count,group_2_count,group_1_median_price,group_2_median_price,median_difference_group_1_minus_group_2,u_statistic,p_value,rank_biserial_correlation,p_value_bonferroni,significant_after_bonferroni
0,Private room,Entire home/apt,1653,4771,171.0,331.0,-160.0,1193572.0,0.000000,-0.697311,0.000000,True
1,Private room,Hotel room,1653,26,171.0,227.5,-56.5,14353.0,0.003626,-0.332077,0.021754,True
2,Private room,Shared room,1653,21,171.0,52.0,119.0,32914.0,0.000000,0.896350,0.000000,True
3,Entire home/apt,Hotel room,4771,26,331.0,227.5,103.5,92913.5,0.000012,0.498049,0.000069,True
4,Entire home/apt,Shared room,4771,21,331.0,52.0,279.0,95458.5,0.000000,0.905530,0.000000,True
5,Hotel room,Shared room,26,21,227.5,52.0,175.5,520.0,0.000000,0.904762,0.000001,True


In [12]:
pairwise_room_type_price_tests_output_path = STAT_REPORTS_PATH / "pairwise_room_type_price_mannwhitney_tests.csv"

pairwise_room_type_price_tests.to_csv(pairwise_room_type_price_tests_output_path, index=False)

print(f"Pairwise room type price tests saved to: {pairwise_room_type_price_tests_output_path}")

Pairwise room type price tests saved to: ..\reports\statistical_analysis\pairwise_room_type_price_mannwhitney_tests.csv


### Pairwise Test Interpretation

The pairwise Mann-Whitney U tests identify which room type pairs have significantly different price distributions.

Bonferroni-adjusted p-values are used because multiple comparisons were performed.  
A pair is treated as statistically significant when the Bonferroni-adjusted p-value is below 0.05.

Small groups such as Hotel room and Shared room should be interpreted cautiously because their sample sizes are much smaller than Entire home/apartment and Private room.

### Pairwise Room Type Price Test Interpretation

The pairwise Mann-Whitney U tests show that all room type pairs have statistically significant price differences after Bonferroni correction.

This confirms that room type is strongly associated with listing price in the Amsterdam Airbnb market.  
Entire home/apartment listings have a much higher median price than private rooms and shared rooms, while shared rooms represent the lowest-priced segment.

Results involving Hotel room and Shared room should be interpreted carefully because these groups have much smaller sample sizes compared with Entire home/apartment and Private room listings.